In [1]:
import os
import ot
import gc
import k3d
import torch
import trimesh
import warnings
from tqdm import *
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from tqdm import tqdm
import seaborn as sns
from pathlib import Path
import scipy.sparse as sp
import matplotlib.cm as cm
from anndata import AnnData
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from numpy.random import RandomState
import matplotlib.font_manager as fm
from matplotlib.gridspec import GridSpec
from sklearn.metrics import jaccard_score
from scipy.stats import fisher_exact, norm
from sklearn.neighbors import NearestNeighbors
from typing import Literal, Optional, Tuple, Union
from matplotlib.colors import ListedColormap, rgb2hex
from sklearn.metrics.pairwise import euclidean_distances
from matplotlib.font_manager import fontManager, FontProperties
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar

import marsilea as ma
import re
from matplotlib_scalebar.scalebar import ScaleBar

plt.rcParams['pdf.fonttype'] = 42
import matplotlib.colors as mcolors
from matplotlib.font_manager import fontManager, FontProperties
import os
fontManager.addfont('/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/Arial.ttf')
font = FontProperties(fname='/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/Arial.ttf')
font_name = font.get_name()
plt.rcParams['font.family'] = font_name


tick_font = FontProperties(fname='/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/Arial-ItalicMT.otf', style = 'italic')

In [2]:
adata_path = "/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/rapids_analysis/merfish_mouseBrain_concat_embeddings.h5ad"
csv_path = "/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/example/05_merfish_mouseBrain/cluster_to_cluster_annotation_membership.csv"
json_path = "/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/rapids_analysis/author_colormaps/author_colormap.json"


adata = sc.read_h5ad(adata_path)
df = pd.read_csv(csv_path)

def strip_author_id(x):
    return re.sub(r"^\s*\d+\s+", "", str(x)).strip()

class_df = df[df["cluster_annotation_term_set_name"] == "class"].copy()
class_df["cluster_alias"] = class_df["cluster_alias"].astype(str)
class_df["author_class"] = class_df["cluster_annotation_term_name"].map(strip_author_id)

cluster_to_class = (
    class_df
    .drop_duplicates("cluster_alias")
    .set_index("cluster_alias")["author_class"]
    .to_dict()
)

adata.obs["author_class"] = (
    adata.obs["cluster_id_transfer"]
    .astype(str)
    .map(cluster_to_class)
    .astype("category")
)

In [3]:
import json

with open(json_path, "r", encoding="utf-8") as f:
    author_cmap = json.load(f)

print(author_cmap.keys())

class_palette = author_cmap["author_term_set_palettes"]["class"]
cluster_palette = author_cmap["obs_key_palettes"]["cluster_id_transfer"]
# subclass_palette = author_cmap["obs_key_palettes"]["subclass_transfer"]
subclass_palette = author_cmap['author_term_set_palettes']['subclass']

cell_leiden_colormap = {
    '11': '#a9889b', '28': '#55f0e9',
 '10': '#03683d', '44': '#68d346', '20': '#063f88', '46': '#aef741', '39': '#e23cd3', '6': '#ddd34f', '53': '#f3e716', '35': '#96b751',
 '30': '#5eed8b', '25': '#13bf07', '42': '#70bed8', '8': '#014d3c', '19': '#169739', '16': '#364fc9', '32': '#49bf04', '18': '#da404e',
 '27': '#3ee944', '52': '#3c2b33', '12': '#329e27', '36': '#4a7e21', '2': '#b8dcb8', '29': '#3ff72d', '9': '#ae3e99', '47': '#1af166',
 '43': '#d6a07b', '50': '#685c07', '48': '#e6e289', '51': '#419ea8', '40': '#1ce209', '22': '#2b968c', '31': '#2b3a18', '49': '#888561',
 '7': '#ce3af9', '4': '#84a490', '23': '#1ed53a', '41': '#d5a20a', '17': '#036f90', '34': '#f97905', '33': '#de0045', '13': '#0f2cec', 
    '38': '#1c2092', '26': '#6b1bfc', '3': '#907d9f',
 '37': '#966844', '45': '#b22f78', '14': '#a0979d', '24': '#10fb09', '5': '#013689', '0': '#19a7b9', '21': '#bea139', '15': '#fd3cdf',
    '1': '#fefa2d'}
major_brain_region_colormap = {
    'Midbrain': '#FFA6FF', 'Cerebellum': '#FFFDBC',
 'Isocortex': '#0D9F91', 'Hippocampus': '#62178d', 'Cortical_subplate': '#97EC93',
 'Medulla': '#FFA6FF', 'Olfactory': '#A8ECD3', 'Fiber_tracts': '#CBCBCB',
 'Thalamus': '#FF909F', 'Hypothalamus': '#F2483B', 'Pallidum': '#B3C0DF',
 'Pons': '#FFA6FF', 'Striatum': '#80C0E2', 'Ventricular_systems': '#AAAAAA', 'n/a': '#8fec63'
}
niche_leiden_colormap = {
    '4': '#24ea8d', '5': '#981cb1',
 '3': '#fc821e', '26': '#518e40', '15': '#47c87b', '17': '#217262', '36': '#9b43a3', '21': '#9d0201', '13': '#33420e', '22': '#93b2a3',
 '23': '#f90830', '25': '#cd94e2', '41': '#5409fe', '0': '#07d774', '42': '#3cdf1a', '48': '#9b50f2', '18': '#b18e8a', '40': '#391dbf',
 '32': '#a97778', '35': '#bb75bb', '6': '#3964ff', '46': '#cce234', '29': '#c73148', '28': '#90fb22', '14': '#022a83', '37': '#649b41',
 '10': '#42b3cc', '19': '#25d0a1', '2': '#bea66e', '38': '#24a568', '45': '#fb08f0', '24': '#598890', '20': '#3f8ee2', '33': '#da6918',
 '43': '#dffbec', '27': '#6e70d2', '11': '#c01c96', '49': '#ea5dd7', '44': '#519017', '1': '#5e319e', '12': '#7de88c', '16': '#3ef055',
 '47': '#2cf922', '30': '#c9af2a', '9': '#803891', '34': '#bf2654', '39': '#83b355', '7': '#1a9bbd', '31': '#06c0bb', '8': '#085d87',
}

donor_id_colormap = {
    'C57BL6J-1': '#73BBF4', 
    'C57BL6J-2': '#284D76', 
    'C57BL6J-3': '#DF95D5', 
    'C57BL6J-4': '#791E25',
}

dict_keys(['source_membership', 'source_h5ad', 'obs_key_palettes', 'author_term_set_palettes'])


In [4]:
import numpy as np
import pandas as pd
from matplotlib.colors import to_rgb, to_hex


def blend_with_white(hex_color, amount=0.0):
    """
    amount=0: 原色
    amount=1: 白色
    """
    rgb = np.array(to_rgb(hex_color))
    white = np.array([1, 1, 1])
    new_rgb = rgb * (1 - amount) + white * amount
    return to_hex(new_rgb)


def build_leiden_palette_from_author(
    adata,
    leiden_key="cell_leiden",
    author_key="author_class",
    class_palette=None,
    min_lighten=0.0,
    max_lighten=0.55,
):
    obs = adata.obs[[leiden_key, author_key]].dropna().copy()

    obs[leiden_key] = obs[leiden_key].astype(str)
    obs[author_key] = obs[author_key].astype(str)

    # Leiden -> author_class composition
    confusion = pd.crosstab(
        obs[leiden_key],
        obs[author_key],
        normalize="index"
    )

    # 每个 Leiden 最相关的 author_class
    best_class = confusion.idxmax(axis=1)
    best_score = confusion.max(axis=1)

    mapping_df = pd.DataFrame({
        "cell_leiden": confusion.index,
        "matched_author_class": best_class.values,
        "matched_fraction": best_score.values,
    })

    leiden_palette = {}

    for author_class, sub_df in mapping_df.groupby("matched_author_class"):
        if class_palette is None or author_class not in class_palette:
            base_color = "#808080"
        else:
            base_color = class_palette[author_class]

        # 同一个 author_class 下，匹配度最高的 Leiden 最接近原色
        sub_df = sub_df.sort_values("matched_fraction", ascending=False)

        n = len(sub_df)

        for rank, (_, row) in enumerate(sub_df.iterrows()):
            leiden = row["cell_leiden"]

            if n == 1:
                lighten = min_lighten
            else:
                lighten = min_lighten + (max_lighten - min_lighten) * rank / (n - 1)

            leiden_palette[leiden] = blend_with_white(base_color, amount=lighten)

    return leiden_palette, mapping_df, confusion

In [5]:
adata = adata[
    adata.obs["major_brain_region"].notna()
    & (adata.obs["major_brain_region"] != "n/a")
].copy()

In [6]:
cell_leiden_palette, leiden_author_map, confusion = build_leiden_palette_from_author(
    adata,
    leiden_key="cell_leiden",
    author_key="author_class",
    class_palette=class_palette,
    min_lighten=0.0,
    max_lighten=0.3,
)

In [7]:
leiden_author_map.to_csv('/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_cell/leiden_author_map.csv')

In [8]:
cell_leiden_palette

{'13': '#594a26',
 '15': '#635533',
 '43': '#6d6040',
 '42': '#776b4d',
 '49': '#81755a',
 '30': '#8b8067',
 '21': '#fffb46',
 '14': '#9d0208',
 '44': '#ba4e52',
 '28': '#90e0ef',
 '50': '#b199ff',
 '32': '#b9a3ff',
 '18': '#c1adff',
 '46': '#c8b8ff',
 '22': '#ccff33',
 '17': '#f954ee',
 '31': '#16f2f2',
 '52': '#ff6600',
 '0': '#ff944c',
 '51': '#01d669',
 '33': '#fa0087',
 '23': '#fa0a8b',
 '37': '#fa1390',
 '45': '#fb1d94',
 '39': '#fb2699',
 '27': '#fb309e',
 '29': '#fb39a2',
 '38': '#fb43a6',
 '10': '#fb4cab',
 '2': '#825f45',
 '53': '#38b000',
 '12': '#9ef01a',
 '5': '#007200',
 '8': '#4c9c4c',
 '35': '#faa307',
 '24': '#61e2a4',
 '40': '#90ebbf',
 '41': '#d00000',
 '16': '#1b4332',
 '48': '#2c5141',
 '26': '#3d5f51',
 '25': '#4e6d60',
 '20': '#5f7b70',
 '36': '#996b2e',
 '4': '#03045e',
 '7': '#292a76',
 '3': '#4f4f8e',
 '47': '#6b5ca5',
 '19': '#0d47a1',
 '11': '#858881',
 '1': '#8e918a',
 '6': '#979a94',
 '34': '#a0a39d',
 '9': '#aaaca7'}

In [9]:
section_slices = ['C57BL6J-1.099', 'C57BL6J-1.080', 'C57BL6J-3.008']
for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'author_class', basis = 'X_spatial_coords', 
                    show=False, s=3, palette = class_palette, frameon = False, title = section_slice, legend_loc = None
                    # groups = ob_cells
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_cell/all/author_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()

In [10]:
section_slices = ['C57BL6J-1.099', 'C57BL6J-1.080', 'C57BL6J-3.008']
for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'cell_leiden', basis = 'X_spatial_coords', 
                    show=False, s=3, palette = cell_leiden_palette, frameon = False, title = section_slice, legend_loc = None
                    # groups = ob_cells
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_cell/all/leiden_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()

In [11]:
section_slices = ['C57BL6J-2.060', 'C57BL6J-3.016', 'C57BL6J-3.004']
for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'author_class', basis = 'X_spatial_coords', 
                    show=False, s=3, palette = class_palette, frameon = False, title = section_slice, legend_loc = None,
                    groups = ['CB Glut', 'CB GABA'],
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_cell/cb_cells/author_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()

In [19]:
section_slices = ['C57BL6J-2.060', 'C57BL6J-3.016', 'C57BL6J-3.004']
for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'cell_leiden', basis = 'X_spatial_coords', 
                    show=False, s=3, palette = cell_leiden_palette, frameon = False, title = section_slice, legend_loc = None,
                    groups = ['14', '21']
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_cell/cb_cells/leiden_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()

In [13]:
section_slices = ['C57BL6J-1.022', 'C57BL6J-1.030', 'C57BL6J-3.006']
for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'author_class', basis = 'X_spatial_coords', 
                    show=False, s=3, palette = class_palette, frameon = False, title = section_slice, legend_loc = None,
                    groups = ['OB-IMN GABA', 'OB-CR Glut'],
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_cell/ob_cells/author_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()

In [14]:
section_slices = ['C57BL6J-1.022', 'C57BL6J-1.030', 'C57BL6J-3.006']
for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'cell_leiden', basis = 'X_spatial_coords', 
                    show=False, s=3, palette = cell_leiden_palette, frameon = False, title = section_slice, legend_loc = None,
                    groups = ['16', '20', '25', '26', '41', '48'],
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_cell/ob_cells/leiden_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()

In [15]:
section_slices = ['C57BL6J-1.080', 'C57BL6J-3.007', 'C57BL6J-3.015']
for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'author_class', basis = 'X_spatial_coords', 
                    show=False, s=3, palette = class_palette, frameon = False, title = section_slice, legend_loc = None,
                    groups = ['IT-ET Glut', 'CTX-MGE GABA', 'CTX-CGE GABA', 'IT-ET Glut', 'NP-CT-L6b Glut'],
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_cell/ctx_cells/author_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()

In [16]:
section_slices = ['C57BL6J-1.080', 'C57BL6J-3.007', 'C57BL6J-3.015']
for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'cell_leiden', basis = 'X_spatial_coords', 
                    show=False, s=3, palette = cell_leiden_palette, frameon = False, title = section_slice, legend_loc = None,
                    groups = ['10', '17', '22', '23', '27', '29', '37', '38', '39', '45', '24', '40'],
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_cell/ctx_cells/leiden_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()

In [17]:
section_slices = ['C57BL6J-1.080', 'C57BL6J-3.007', 'C57BL6J-3.015']
for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'author_class', basis = 'X_spatial_coords', 
                    show=False, s=3, palette = class_palette, frameon = False, title = section_slice, legend_loc = None,
                    groups = ['Astro-Epen', 'OPC-Oligo', 'Vascular'],
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_cell/nn_cells/author_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()

In [18]:
section_slices = ['C57BL6J-1.080', 'C57BL6J-3.007', 'C57BL6J-3.015']
for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'cell_leiden', basis = 'X_spatial_coords', 
                    show=False, s=3, palette = cell_leiden_palette, frameon = False, title = section_slice, legend_loc = None,
                    groups = ['1', '11', '34', '6', '9', '13', '15', '30', '42', '43', '49', '3', '4', '7'],
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_cell/nn_cells/leiden_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()